# Lexical Feature Extraction Pipeline

**Goal:** Extract linguistic features from group conversation transcripts for HMM integration.

**Date:** 2026-08-13

---

## Overview

This notebook documents the lexical feature extraction process:
1. **Participant-level features** (32 features per participant × task)
2. **Window-level features** (15 features per 30s window) for HMM integration
3. **Theory-driven composites** (social, complexity, verbosity)

## Data Source

Transcripts from speaker diarization: `transcripts/{group_id}/{task_id}_diarized.tsv`

Each row has: `onset`, `offset`, `speaker`, `label`, `text`

---
## 1. Setup & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# NLTK for lemmatization
import nltk
from nltk.stem import WordNetLemmatizer
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

# Paths
TRANSCRIPT_DIR = Path('../../transcripts')
RESULTS_DIR = Path('../../analysis/results')
RESULTS_DIR.mkdir(exist_ok=True)

# Groups and tasks
GROUPS = [f'grp-{i:02d}' for i in range(7, 17)]
TASKS = ['T1', 'T2', 'T3']

print(f'✓ Libraries loaded')
print(f'Groups: {GROUPS}')
print(f'Tasks: {TASKS}')

---
## 2. Word Lists (Linguistic Markers)

We define word lists for counting specific linguistic phenomena.
All words are **base forms** (lemmas) to handle conjugation (e.g., "agreeing" → "agree").

In [ ]:
# Agreement markers (base forms)
AGREEMENT_WORDS = {
    'yes', 'yeah', 'right', 'exactly', 'agree', 'absolutely', 'definitely',
    'correct', 'sure', 'okay', 'ok', 'good', 'great', 'perfect', 'fine',
    'indeed', 'precisely', 'totally', 'certainly', 'true', 'yep', 'yup'
}

# Positive sentiment words
POSITIVE_WORDS = {
    'good', 'great', 'excellent', 'nice', 'wonderful', 'fantastic', 'amazing',
    'awesome', 'brilliant', 'perfect', 'love', 'like', 'enjoy', 'happy',
    'pleased', 'glad', 'excited', 'interesting', 'helpful', 'useful',
    'agree', 'best', 'better', 'benefit', 'success', 'successful',
    'effective', 'efficient'
}

# Negative sentiment words
NEGATIVE_WORDS = {
    'bad', 'wrong', 'terrible', 'awful', 'horrible', 'poor', 'worst',
    'hate', 'dislike', 'boring', 'annoying', 'frustrating', 'difficult',
    'hard', 'problem', 'issue', 'fail', 'failure', 'mistake', 'error',
    'disagree', 'no', 'not', 'never', 'cannot', "can't", "don't", "won't"
}

# Hedging markers
HEDGING_WORDS = {
    'maybe', 'perhaps', 'possibly', 'probably', 'might', 'could', 'would',
    'kind', 'sort', 'somewhat', 'fairly', 'rather', 'quite', 'basically',
    'actually', 'just', 'like', 'guess', 'think', 'believe', 'suppose',
    'seem', 'apparently', 'presumably'
}

# Certainty markers
CERTAINTY_WORDS = {
    'definitely', 'certainly', 'absolutely', 'clearly', 'obviously',
    'surely', 'undoubtedly', 'indeed', 'always', 'must', 'will',
    'know', 'sure', 'certain', 'confident'
}

# Question markers
QUESTION_WORDS = {
    'what', 'why', 'how', 'when', 'where', 'who', 'which', 'whom',
    'whose', '?'
}

# Suggestion markers
SUGGESTION_WORDS = {
    'should', 'could', 'would', 'suggest', 'propose', 'recommend',
    'idea', 'think', 'maybe', 'perhaps', 'let', "let's", 'why'
}

print(f'Agreement words: {len(AGREEMENT_WORDS)}')
print(f'Positive words: {len(POSITIVE_WORDS)}')
print(f'Negative words: {len(NEGATIVE_WORDS)}')
print(f'Hedging words: {len(HEDGING_WORDS)}')
print(f'Certainty words: {len(CERTAINTY_WORDS)}')

---
## 3. Text Processing Functions

In [ ]:
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def tokenize(text: str) -> list[str]:
    """Simple whitespace + punctuation tokenizer."""
    if pd.isna(text) or not isinstance(text, str):
        return []
    text = text.lower()
    tokens = re.findall(r"[a-z']+|\?", text)
    return tokens

def lemmatize_word(word: str) -> str:
    """Lemmatize a word (verb-first, noun-fallback)."""
    lemma_v = lemmatizer.lemmatize(word, pos='v')
    if lemma_v != word:
        return lemma_v
    return lemmatizer.lemmatize(word, pos='n')

def count_markers(tokens: list[str], marker_set: set) -> int:
    """Count tokens that match marker set (after lemmatization)."""
    return sum(1 for t in tokens if lemmatize_word(t) in marker_set)

def get_ngrams(tokens: list[str], n: int) -> list[tuple]:
    """Generate n-grams from token list."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def ngram_entropy(tokens: list[str], n: int) -> float:
    """Compute normalized entropy of n-gram distribution."""
    ngrams = get_ngrams(tokens, n)
    if len(ngrams) < 2:
        return 0.0
    counts = Counter(ngrams)
    total = sum(counts.values())
    probs = np.array([c / total for c in counts.values()])
    entropy = -np.sum(probs * np.log2(probs + 1e-10))
    max_entropy = np.log2(len(counts))
    return entropy / max_entropy if max_entropy > 0 else 0.0

# Test
test_text = "I think we should agree on this. It's a great idea!"
tokens = tokenize(test_text)
print(f'Original: {test_text}')
print(f'Tokens: {tokens}')
print(f'Lemmas: {[lemmatize_word(t) for t in tokens]}')
print(f'Agreement count: {count_markers(tokens, AGREEMENT_WORDS)}')
print(f'Positive count: {count_markers(tokens, POSITIVE_WORDS)}')

---
## 4. Extract Participant-Level Features

For each **participant × task**, we compute 32 lexical features.

In [ ]:
def extract_participant_features(group_id: str, task_id: str, participant: str, df: pd.DataFrame) -> dict:
    """Extract lexical features for one participant in one task."""
    # Filter to SPK rows for this participant
    mask = (df['speaker'] == participant) & (df['label'] == 'SPK')
    rows = df[mask]
    
    # Concatenate all text
    all_text = ' '.join(rows['text'].dropna().astype(str))
    tokens = tokenize(all_text)
    
    # Basic counts
    word_count = len(tokens)
    unique_words = len(set(tokens))
    utterance_count = len(rows)
    
    # Derived metrics
    ttr = unique_words / word_count if word_count > 0 else 0
    words_per_utterance = word_count / utterance_count if utterance_count > 0 else 0
    
    # N-gram entropy
    bigram_entropy = ngram_entropy(tokens, 2)
    trigram_entropy = ngram_entropy(tokens, 3)
    
    # Marker counts
    agreement_count = count_markers(tokens, AGREEMENT_WORDS)
    hedging_count = count_markers(tokens, HEDGING_WORDS)
    certainty_count = count_markers(tokens, CERTAINTY_WORDS)
    positive_count = count_markers(tokens, POSITIVE_WORDS)
    negative_count = count_markers(tokens, NEGATIVE_WORDS)
    question_count = count_markers(tokens, QUESTION_WORDS)
    suggestion_count = count_markers(tokens, SUGGESTION_WORDS)
    
    # Ratios (safe division)
    def safe_ratio(num, denom):
        return num / denom if denom > 0 else 0
    
    agreement_ratio = safe_ratio(agreement_count, word_count)
    hedging_ratio = safe_ratio(hedging_count, word_count)
    certainty_ratio = safe_ratio(certainty_count, word_count)
    sentiment_ratio = safe_ratio(positive_count - negative_count, word_count)
    question_ratio = safe_ratio(question_count, word_count)
    
    return {
        'group_id': group_id,
        'task_id': task_id,
        'participant_id': participant,
        'word_count': word_count,
        'unique_words': unique_words,
        'utterance_count': utterance_count,
        'ttr': ttr,
        'words_per_utterance': words_per_utterance,
        'bigram_entropy': bigram_entropy,
        'trigram_entropy': trigram_entropy,
        'agreement_count': agreement_count,
        'hedging_count': hedging_count,
        'certainty_count': certainty_count,
        'positive_count': positive_count,
        'negative_count': negative_count,
        'question_count': question_count,
        'suggestion_count': suggestion_count,
        'agreement_ratio': agreement_ratio,
        'hedging_ratio': hedging_ratio,
        'certainty_ratio': certainty_ratio,
        'sentiment_ratio': sentiment_ratio,
        'question_ratio': question_ratio,
    }

print('Feature extraction function defined')

In [ ]:
# Extract features for all participants
all_features = []

for group in GROUPS:
    for task in TASKS:
        # Load transcript
        transcript_path = TRANSCRIPT_DIR / group / f'{task}_diarized.tsv'
        if not transcript_path.exists():
            continue
        
        df = pd.read_csv(transcript_path, sep='\t')
        
        # Get unique participants (P1-P4)
        participants = sorted(df['speaker'].dropna().unique())
        participants = [p for p in participants if p in ['P1', 'P2', 'P3', 'P4']]
        
        for participant in participants:
            features = extract_participant_features(group, task, participant, df)
            all_features.append(features)

# Create dataframe
participant_df = pd.DataFrame(all_features)
print(f'Extracted features for {len(participant_df)} participant × task instances')
print(f'Groups: {participant_df["group_id"].nunique()}')
print(f'Tasks: {participant_df["task_id"].nunique()}')
print(f'Columns: {len(participant_df.columns)}')

# Show sample
participant_df.head()

In [ ]:
# Save participant-level features
participant_df.to_csv(RESULTS_DIR / 'participant_lexical_features.tsv', sep='\t', index=False)
print(f'Saved: {RESULTS_DIR / "participant_lexical_features.tsv"}')

# Summary statistics
print('\nFeature statistics:')
print(participant_df.describe().round(2))

---
## 5. Extract Window-Level Features (30s windows)

For HMM integration, we need features at the **30-second window** level.

We aggregate utterances within each window and compute group-level statistics.

In [ ]:
WINDOW_SIZE_S = 30  # 30-second windows

def extract_window_features(group_id: str, task_id: str, df: pd.DataFrame) -> list[dict]:
    """Extract lexical features for each 30s window in a task."""
    # Filter to SPK rows only
    spk_df = df[df['label'] == 'SPK'].copy()
    if len(spk_df) == 0:
        return []
    
    # Assign window index based on onset
    spk_df['window_index'] = (spk_df['onset'] // WINDOW_SIZE_S).astype(int)
    
    results = []
    for window_idx, window_df in spk_df.groupby('window_index'):
        # Concatenate all text in window
        all_text = ' '.join(window_df['text'].dropna().astype(str))
        tokens = tokenize(all_text)
        
        word_count = len(tokens)
        unique_words = len(set(tokens))
        
        # Skip windows with no words (will be imputed later)
        ttr = unique_words / word_count if word_count > 0 else np.nan
        bigram_entropy = ngram_entropy(tokens, 2) if word_count >= 3 else np.nan
        trigram_entropy = ngram_entropy(tokens, 3) if word_count >= 4 else np.nan
        
        # Counts
        agreement_count = count_markers(tokens, AGREEMENT_WORDS)
        hedging_count = count_markers(tokens, HEDGING_WORDS)
        certainty_count = count_markers(tokens, CERTAINTY_WORDS)
        positive_count = count_markers(tokens, POSITIVE_WORDS)
        negative_count = count_markers(tokens, NEGATIVE_WORDS)
        question_count = count_markers(tokens, QUESTION_WORDS)
        suggestion_count = count_markers(tokens, SUGGESTION_WORDS)
        
        # Ratios
        sentiment_ratio = (positive_count - negative_count) / word_count if word_count > 0 else 0
        
        # Social composite (z-scored average of agreement + positive + hedging)
        # Will be normalized later at dataset level
        social_composite = agreement_count + positive_count + hedging_count
        
        results.append({
            'group_id': group_id,
            'task_id': task_id,
            'window_index': int(window_idx),
            'window_start_s': window_idx * WINDOW_SIZE_S,
            'lex_word_count': word_count,
            'lex_unique_words': unique_words,
            'lex_ttr': ttr,
            'lex_bigram_entropy': bigram_entropy,
            'lex_trigram_entropy': trigram_entropy,
            'lex_agreement_count': agreement_count,
            'lex_hedging_count': hedging_count,
            'lex_certainty_count': certainty_count,
            'lex_positive_count': positive_count,
            'lex_negative_count': negative_count,
            'lex_question_count': question_count,
            'lex_suggestion_count': suggestion_count,
            'lex_sentiment_ratio': sentiment_ratio,
            'lex_social_composite': social_composite,
        })
    
    return results

print('Window feature extraction function defined')

In [ ]:
# Extract window features for all groups and tasks
all_window_features = []

for group in GROUPS:
    for task in TASKS:
        transcript_path = TRANSCRIPT_DIR / group / f'{task}_diarized.tsv'
        if not transcript_path.exists():
            continue
        
        df = pd.read_csv(transcript_path, sep='\t')
        features = extract_window_features(group, task, df)
        all_window_features.extend(features)

# Create dataframe
window_df = pd.DataFrame(all_window_features)
print(f'Extracted features for {len(window_df)} windows')
print(f'Groups: {window_df["group_id"].nunique()}')
print(f'Tasks: {window_df["task_id"].nunique()}')

# Handle NaN values (windows with zero words)
nan_cols = ['lex_ttr', 'lex_bigram_entropy', 'lex_trigram_entropy']
for col in nan_cols:
    n_nan = window_df[col].isna().sum()
    print(f'{col}: {n_nan} NaN values → imputed to 0')
    window_df[col] = window_df[col].fillna(0)

window_df.head()

In [ ]:
# Save window-level features
window_df.to_csv(RESULTS_DIR / 'lexical_window_30s.tsv', sep='\t', index=False)
print(f'Saved: {RESULTS_DIR / "lexical_window_30s.tsv"}')

# Summary
print('\nFeature statistics:')
print(window_df.describe().round(2))

---
## 6. Feature Distributions

In [ ]:
# Plot distributions of key features
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

features_to_plot = [
    'lex_word_count', 'lex_unique_words', 'lex_ttr', 'lex_bigram_entropy',
    'lex_agreement_count', 'lex_positive_count', 'lex_hedging_count', 'lex_sentiment_ratio'
]

for i, feat in enumerate(features_to_plot):
    ax = axes[i]
    window_df[feat].hist(bins=30, ax=ax, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(feat.replace('lex_', ''), fontsize=11)
    ax.set_xlabel('')

plt.suptitle('Lexical Feature Distributions (30s windows)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlations
lex_cols = [c for c in window_df.columns if c.startswith('lex_')]
corr_matrix = window_df[lex_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Lexical Feature Correlations', fontsize=14)
plt.tight_layout()
plt.show()

---
## 7. Task-Level Analysis

In [ ]:
# Compare features across tasks
task_summary = window_df.groupby('task_id')[lex_cols].mean()
print('Mean lexical features by task:')
print(task_summary.round(2).T)

In [ ]:
# Visualize task differences
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Word count
sns.boxplot(data=window_df, x='task_id', y='lex_word_count', palette='Set2', ax=axes[0])
axes[0].set_title('Word Count by Task')
axes[0].set_xlabel('Task')
axes[0].set_ylabel('Words per 30s window')

# Agreement
sns.boxplot(data=window_df, x='task_id', y='lex_agreement_count', palette='Set2', ax=axes[1])
axes[1].set_title('Agreement Markers by Task')
axes[1].set_xlabel('Task')
axes[1].set_ylabel('Count per 30s window')

# Sentiment
sns.boxplot(data=window_df, x='task_id', y='lex_sentiment_ratio', palette='Set2', ax=axes[2])
axes[2].set_title('Sentiment Ratio by Task')
axes[2].set_xlabel('Task')
axes[2].set_ylabel('(Positive - Negative) / Words')

plt.tight_layout()
plt.show()

---
## 8. Summary

### Output Files

| File | Level | Rows | Features |
|------|-------|------|----------|
| `participant_lexical_features.tsv` | Participant × Task | ~148 | 22 |
| `lexical_window_30s.tsv` | Window (30s) | ~682 | 15 |

### Key Features for HMM

The following features are integrated into the HMM:
- `lex_word_count` — Total words (verbosity)
- `lex_agreement_count` — Agreement markers
- `lex_positive_count` — Positive sentiment words
- `lex_sentiment_ratio` — Net sentiment
- `lex_social_composite` — Combined social markers

### Lemmatization

All marker matching uses **lemmatized forms** to handle verb conjugations:
- "agreeing" → "agree" ✓
- "agreed" → "agree" ✓
- "thinks" → "think" ✓

In [ ]:
print('='*60)
print('LEXICAL FEATURE EXTRACTION COMPLETE')
print('='*60)
print(f'Participant-level: {len(participant_df)} rows × {len(participant_df.columns)} cols')
print(f'Window-level: {len(window_df)} rows × {len(window_df.columns)} cols')
print('='*60)